# Solutions Notebook | Business Applications in AI

This notebook contains reference solutions for the Day 1 and Day 2 exercises.
These represent strong answer shapes, not the only correct answers.

**For facilitator use only.** Do not distribute to participants before submission.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
opportunities = pd.read_csv(DATA_DIR / "ai_opportunities.csv")
governance = pd.read_csv(DATA_DIR / "governance_checklist.csv")

## Day 1 Solution: Opportunity Scoring and Ranking

The solution below demonstrates score adjustments with rationale,
composite scoring, top-3 selection, and a park recommendation.

In [ ]:
# Score adjustments with rationale
adjustments = {
    "OPP-004": {"feasibility_score": 4, "rationale": "Bank already has transaction data infrastructure; raised from 3 to 4"},
    "OPP-005": {"risk_score": 5, "rationale": "SAR narratives are regulatory filings; generative AI risk is very high; raised from 4 to 5"},
    "OPP-007": {"feasibility_score": 1, "rationale": "Data only partial, no labelled training set for edge cases; reduced from 2 to 1"},
    "OPP-010": {"feasibility_score": 3, "rationale": "Alternative data sources exist (transaction behaviour); raised from 2 to 3"},
    "OPP-012": {"feasibility_score": 1, "rationale": "No structured data available for regulatory change analysis; reduced from 2 to 1"},
    "OPP-020": {"feasibility_score": 1, "rationale": "No call recording data pipeline in place; reduced from 2 to 1"},
}

opportunities["score_rationale"] = ""
for opp_id, adj in adjustments.items():
    mask = opportunities["opportunity_id"] == opp_id
    for col, val in adj.items():
        if col != "rationale":
            opportunities.loc[mask, col] = val
    opportunities.loc[mask, "score_rationale"] = adj.get("rationale", "")

# Composite priority score
opportunities["priority_score"] = (
    opportunities["value_score"]
    + opportunities["feasibility_score"]
    - opportunities["risk_score"]
)

ranked = opportunities.sort_values("priority_score", ascending=False)
ranked[["opportunity_id", "department", "workflow", "value_score", "feasibility_score", "risk_score", "priority_score", "score_rationale"]].head(10)

In [ ]:
# Top 3 selection with justification
top3 = {
    "OPP-001": "Complaint classification scores high on value (SAR 1.2M saving) and feasibility (3 years of labelled data). Risk is low because the model suggests, a human confirms. This builds NLP capability reusable across other workflows.",
    "OPP-008": "Trade finance document extraction addresses a clear, measurable pain point (30 min per document). Data is available and structured. Internal workflow with human verification keeps risk low.",
    "OPP-004": "Transaction monitoring false positive reduction has the highest estimated saving (SAR 2.1M). Feasibility is strong given existing data infrastructure. Risk is moderate but managed through human investigation of all alerts.",
}

# Park recommendation
park = {
    "OPP-007": "Customer inquiry chatbot has high value potential but very high risk (direct customer interaction) and low feasibility (partial data, no labelled edge cases). Park until: (1) a comprehensive training dataset is built from call centre transcripts, (2) a confidence threshold and escalation path are designed, (3) SAMA guidance on customer-facing AI is clarified.",
}

opportunities["status"] = "other"
for opp_id in top3:
    opportunities.loc[opportunities["opportunity_id"] == opp_id, "status"] = "top3"
for opp_id in park:
    opportunities.loc[opportunities["opportunity_id"] == opp_id, "status"] = "park"

opportunities["justification"] = ""
for opp_id, text in {**top3, **park}.items():
    opportunities.loc[opportunities["opportunity_id"] == opp_id, "justification"] = text

print("=== Top 3 ===")
for opp_id, text in top3.items():
    print(f"\n{opp_id}: {text}")

print("\n=== Park ===")
for opp_id, text in park.items():
    print(f"\n{opp_id}: {text}")

## Day 2 Solution: Governance Assessment

Assessment of OPP-001 (Complaint classification) against the governance checklist.
This demonstrates how to evaluate each control for a specific opportunity.

In [ ]:
# Governance assessment for OPP-001: Complaint classification and routing
gov_assessment = {
    "GOV-001": {"status": "Met", "action": "", "owner": "", "timeline": ""},
    "GOV-002": {"status": "Partially Met", "action": "Define formal escalation path for misclassification disputes", "owner": "Operations Manager", "timeline": "2 weeks"},
    "GOV-003": {"status": "Met", "action": "", "owner": "", "timeline": ""},
    "GOV-004": {"status": "Not Met", "action": "Implement logging of all AI classification suggestions and agent confirmations", "owner": "Technology Lead", "timeline": "4 weeks"},
    "GOV-005": {"status": "Not Applicable", "action": "", "owner": "", "timeline": ""},
    "GOV-006": {"status": "Not Applicable", "action": "", "owner": "", "timeline": ""},
    "GOV-007": {"status": "Partially Met", "action": "Audit CRM complaint data for label consistency over 3-year period", "owner": "Data Quality Lead", "timeline": "3 weeks"},
    "GOV-008": {"status": "Met", "action": "", "owner": "", "timeline": ""},
    "GOV-009": {"status": "Not Met", "action": "Complete model risk classification per MRM policy", "owner": "Model Risk Manager", "timeline": "2 weeks"},
    "GOV-010": {"status": "Not Met", "action": "Define quarterly retraining schedule and monthly accuracy monitoring", "owner": "Model Owner", "timeline": "2 weeks"},
    "GOV-011": {"status": "Partially Met", "action": "Document manual triage fallback procedure with SLA for AI downtime", "owner": "Operations Manager", "timeline": "1 week"},
    "GOV-012": {"status": "Not Met", "action": "Configure accuracy and latency monitoring with alerting thresholds", "owner": "Technology Lead", "timeline": "3 weeks"},
    "GOV-013": {"status": "Met", "action": "", "owner": "", "timeline": ""},
    "GOV-014": {"status": "Not Applicable", "action": "", "owner": "", "timeline": ""},
    "GOV-015": {"status": "Not Applicable", "action": "", "owner": "", "timeline": ""},
}

governance["status"] = governance["control_id"].map(lambda x: gov_assessment[x]["status"])
governance["remediation_action"] = governance["control_id"].map(lambda x: gov_assessment[x]["action"])
governance["remediation_owner"] = governance["control_id"].map(lambda x: gov_assessment[x]["owner"])
governance["remediation_timeline"] = governance["control_id"].map(lambda x: gov_assessment[x]["timeline"])

governance

In [ ]:
# Top 3 governance gaps
print("Top 3 Governance Gaps for OPP-001:")
print()
print("1. GOV-004 (Audit trail): No logging of AI suggestions and agent decisions.")
print("   Risk: Cannot demonstrate compliance or investigate misclassification patterns.")
print()
print("2. GOV-010 (Retraining schedule): No defined schedule for model retraining or monitoring.")
print("   Risk: Model accuracy will degrade over time as complaint patterns change.")
print()
print("3. GOV-012 (Performance monitoring): No thresholds or alerting configured.")
print("   Risk: Performance degradation may go undetected until business impact is visible.")
print()
print("Readiness summary:")
print("The proposed pilot for complaint classification is partially ready from a")
print("governance perspective. Three controls require remediation before launch:")
print("audit trail implementation, retraining schedule definition, and performance")
print("monitoring setup. The most critical gap is the audit trail (GOV-004) because")
print("without it, the bank cannot demonstrate oversight or investigate classification")
print("errors. We recommend a 4-week governance preparation phase before the")
print("technical pilot begins.")

In [ ]:
# Value case calculation for OPP-001
daily_volume = 800
time_saved_minutes = 4
working_days = 250
hourly_cost_sar = 120

hours_saved = (daily_volume * time_saved_minutes * working_days) / 60
annual_saving = hours_saved * hourly_cost_sar
conservative = (daily_volume * (time_saved_minutes * 0.75) * working_days) / 60 * hourly_cost_sar

print(f"Hours saved per year: {hours_saved:,.0f}")
print(f"Annual saving: SAR {annual_saving:,.0f}")
print(f"Conservative (75% of time saved): SAR {conservative:,.0f}")
print(f"Value case range: SAR {conservative:,.0f} to SAR {annual_saving:,.0f}")
print()
print("Key assumptions:")
print("- 800 complaints/day is the current average (validate with CX Operations)")
print("- 4 minutes saved assumes the model classifies correctly 85%+ of the time")
print("- Fully loaded cost of SAR 120/hour includes salary, benefits, and overhead")
print("- 250 working days accounts for weekends and public holidays")